# Notebook 12: Trajectory Correction

**R5 — Drift Correction via Predicted Error Subtraction**

This notebook demonstrates the practical payoff of drift prediction:
we subtract the predicted 3-axis error from each window's dead-reckoned endpoint
and reconstruct corrected flight paths for MH_01_easy and MH_02_easy.

In [ ]:
import sys
from pathlib import Path
root = Path('.').resolve().parent
sys.path.insert(0, str(root / 'src'))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

TRAJ_DIR = root / 'results' / 'trajectories'
CORR_CSV = root / 'results' / 'ablations' / 'correction_c1.csv'
FIG_DIR  = root / 'figures' / 'trajectories'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Trajectory dir:', TRAJ_DIR)


## Section 1: Correction Summary

Load the correction metrics table from `results/ablations/correction_c1.csv`.

In [ ]:
df_corr = pd.read_csv(CORR_CSV)
cols = ['sequence','windows_count','endpoint_mae_before_cm','endpoint_mae_after_cm',
        'endpoint_reduction_pct','cumulative_ate_raw_m','cumulative_ate_corr_m',
        'final_drift_raw_m','final_drift_corr_m','ate_reduction_pct']
print(df_corr[cols].to_string(index=False))
for _, row in df_corr.iterrows():
    print(f"\n{row['sequence']}:")
    print(f"  Endpoint MAE: {row['endpoint_mae_before_cm']:.1f} -> {row['endpoint_mae_after_cm']:.1f} cm "
          f"({row['endpoint_reduction_pct']:.1f}% reduction)")
    print(f"  Final drift: {row['final_drift_raw_m']:.2f} -> {row['final_drift_corr_m']:.3f} m "
          f"({row['ate_reduction_pct']:.1f}% ATE reduction)")


## Section 2: MH_01_easy Trajectory

XY path and ATE over time for sequence MH_01_easy.

In [ ]:
df1 = pd.read_csv(TRAJ_DIR / 'trajectory_MH_01_easy.csv')
print(f'MH_01_easy: {len(df1)} non-overlapping window steps')
print(df1[['step','ate_raw','ate_corr']].describe())


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# XY path
ax1.plot(df1['true_x'], df1['true_y'], 'g-', linewidth=2.0, label='Ground truth', zorder=4)
ax1.plot(df1['raw_x'],  df1['raw_y'],  'r--', linewidth=1.2, alpha=0.7, label='Dead-reckoned (raw)', zorder=3)
ax1.plot(df1['corr_x'], df1['corr_y'], 'b-',  linewidth=1.5, label='Corrected', zorder=4)
ax1.scatter([df1['true_x'].iloc[0]], [df1['true_y'].iloc[0]], c='g', s=80, marker='^', zorder=5)
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)')
ax1.set_title('MH_01_easy: XY Trajectory')
ax1.legend(fontsize=9); ax1.grid(alpha=0.4); ax1.set_aspect('equal')

# ATE vs step
ax2.plot(df1['step'], df1['ate_raw'],  'r-', linewidth=1.5, label='ATE raw',       alpha=0.8)
ax2.plot(df1['step'], df1['ate_corr'], 'b-', linewidth=1.5, label='ATE corrected', alpha=0.8)
ax2.set_xlabel('Step'); ax2.set_ylabel('ATE (m)')
ax2.set_title('MH_01_easy: ATE vs Step')
ax2.legend(fontsize=9); ax2.grid(alpha=0.4)
plt.suptitle('MH_01_easy Trajectory Correction', fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Final ATE raw={df1['ate_raw'].iloc[-1]:.3f}m  corrected={df1['ate_corr'].iloc[-1]:.3f}m")


## Section 3: MH_02_easy Trajectory

In [ ]:
df2 = pd.read_csv(TRAJ_DIR / 'trajectory_MH_02_easy.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(df2['true_x'], df2['true_y'], 'g-', linewidth=2.0, label='Ground truth', zorder=4)
ax1.plot(df2['raw_x'],  df2['raw_y'],  'r--', linewidth=1.2, alpha=0.7, label='Dead-reckoned (raw)', zorder=3)
ax1.plot(df2['corr_x'], df2['corr_y'], 'b-',  linewidth=1.5, label='Corrected', zorder=4)
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)')
ax1.set_title('MH_02_easy: XY Trajectory')
ax1.legend(fontsize=9); ax1.grid(alpha=0.4); ax1.set_aspect('equal')
ax2.plot(df2['step'], df2['ate_raw'],  'r-', linewidth=1.5, label='ATE raw',       alpha=0.8)
ax2.plot(df2['step'], df2['ate_corr'], 'b-', linewidth=1.5, label='ATE corrected', alpha=0.8)
ax2.set_xlabel('Step'); ax2.set_ylabel('ATE (m)')
ax2.set_title('MH_02_easy: ATE vs Step')
ax2.legend(fontsize=9); ax2.grid(alpha=0.4)
plt.suptitle('MH_02_easy Trajectory Correction', fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Final ATE raw={df2['ate_raw'].iloc[-1]:.3f}m  corrected={df2['ate_corr'].iloc[-1]:.3f}m")


## Section 4: Side-by-side Comparison — ATE over Time

Both sequences overlaid: raw drift (dashed) vs corrected (solid).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df1['step'], df1['ate_raw'],  color='#E53935', linewidth=1.4, linestyle='--', label='MH_01 raw')
ax.plot(df1['step'], df1['ate_corr'], color='#1E88E5', linewidth=1.8,                 label='MH_01 corrected')
ax.plot(df2['step'], df2['ate_raw'],  color='#FB8C00', linewidth=1.4, linestyle='--', label='MH_02 raw')
ax.plot(df2['step'], df2['ate_corr'], color='#43A047', linewidth=1.8,                 label='MH_02 corrected')
ax.set_xlabel('Window step index'); ax.set_ylabel('ATE (m)')
ax.set_title('ATE over Time — Before vs After Drift Correction')
ax.legend(fontsize=9); ax.grid(alpha=0.4); plt.tight_layout(); plt.show()


## Section 5: Correction Method Summary

**Method:** For each non-overlapping window (every 8th, 2.0 s stride at 0.25 s step):
1. Physics prior predicts 3-axis drift vector [Δx, Δy, Δz]
2. Corrected endpoint = dead-reckoned endpoint − predicted drift
3. Reconstruct continuous path by chaining corrected endpoints

**Results summary:**

| Sequence | Windows | Endpoint MAE Before | Endpoint MAE After | Reduction | ATE Reduction |
|---|---|---|---|---|---|
| MH_01_easy | 90 | 121.5 cm | 6.9 cm | 94.3% | 98.0% |
| MH_02_easy | 74 | 120.0 cm | 5.3 cm | 95.6% | 99.0% |

**Conclusion:** Drift correction reduces absolute trajectory error by >98%, demonstrating clear practical value.